# LSTM with the same grouped folds as `scripts/04_cross_validate.py`

Run on Colab (a CPU runtime is enough, the model is tiny). Upload these four files to `/content` with the upload cell, or mount Drive and point `DATA` at a folder holding them:

- `config.yaml`
- `data/labels.csv` as `labels.csv`
- `data/features/windows.parquet` as `windows.parquet`
- `data/features/sequences.npz` as `sequences.npz`

Leakage controls:

- Only `datasets.train` rows are kept. CAUCAFall rows are dropped before anything else and never reach a fit, a threshold or a choice.
- Outer folds use the same `StratifiedGroupKFold` call, rows and groups as `src/evaluate.outer_splits`. Compare the printed fold table with the one printed by `scripts/04_cross_validate.py`; they must match.
- Inside each training fold, one split of whole clips is held out as validation. It drives early stopping, the choice of units and the recall target threshold. The test fold is touched only once, after the threshold is frozen.
- Augmentation (horizontal flip with left and right joints swapped, plus small noise) is applied to the fitting part of the training fold only.

LSTM settings are read from `models.lstm` in `config.yaml`. The notebook stops with an error when that block or any of its keys is missing.

Output: `lstm_cv_results.csv` with the same columns as `results/tables/cv_results.csv` (detector `lstm`, variants `default` and `tuned`). Download it and keep it as its own file, `results/tables/lstm_cv_results.csv`. Do not merge it into `cv_results.csv`, because `scripts/04_cross_validate.py` overwrites that file.

In [ ]:
from pathlib import Path

import keras
import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
import yaml
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold

DATA = Path("/content")
LSTM_KEYS = ["units_grid", "default_units", "dropout", "learning_rate", "epochs", "patience", "batch_size", "val_splits", "noise_std"]
print("tensorflow", tf.__version__, "keras", keras.__version__, "sklearn", sklearn.__version__)

Upload the four files. Skip this cell when `DATA` points at a mounted Drive folder.

In [ ]:
from google.colab import files

uploaded = files.upload()
print(sorted(uploaded))

In [ ]:
cfg = yaml.safe_load((DATA / "config.yaml").read_text(encoding="utf-8"))
EVAL = cfg["evaluation"]
SEED = EVAL["seed"]
if "lstm" not in cfg["models"]:
    raise KeyError("config.yaml has no models.lstm block, add it before running this notebook")
LSTM = cfg["models"]["lstm"]
absent_keys = [key for key in LSTM_KEYS if key not in LSTM]
if absent_keys:
    raise KeyError(f"models.lstm in config.yaml is missing {absent_keys}")
if LSTM["default_units"] not in LSTM["units_grid"]:
    raise ValueError("default_units must be one of units_grid")
keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

all_windows = pd.read_parquet(DATA / "windows.parquet")
sequences = np.load(DATA / "sequences.npz")
labels = pd.read_csv(DATA / "labels.csv")
if len(sequences["clip_id"]) != len(all_windows):
    raise ValueError("sequences.npz and windows.parquet have different row counts")
same_order = (sequences["clip_id"] == all_windows["clip_id"].to_numpy()) & (sequences["dataset"] == all_windows["dataset"].to_numpy())
if not same_order.all():
    raise ValueError("sequences.npz and windows.parquet are not in the same row order")

keep = all_windows["dataset"].isin(cfg["datasets"]["train"]).to_numpy()
windows = all_windows[keep].reset_index(drop=True)
leaked = set(windows["dataset"]) & set(cfg["datasets"]["test"])
if leaked:
    raise ValueError(f"test datasets in training windows: {sorted(leaked)}")
X = sequences["X"][keep].astype(np.float32)
missing = sequences["missing"][keep] if "missing" in sequences.files else np.zeros(X.shape, dtype=bool)
y = windows["label"].to_numpy().astype(int)
if not np.array_equal(y, sequences["y"][keep]):
    raise ValueError("labels differ between sequences.npz and windows.parquet")
if X.shape[-1] != 24:
    raise ValueError(f"expected 24 channels (12 joints x and y), got {X.shape[-1]}")

train_labels = labels[labels["dataset"].isin(cfg["datasets"]["train"])].set_index("clip_id")
truth = train_labels["fall_start_s"].notna().astype(int)
absent = set(windows["clip_id"]) - set(truth.index)
if absent:
    raise ValueError(f"{len(absent)} window clips missing from labels.csv")

print("training windows per dataset and class:")
print(windows.groupby(["dataset", "label"]).size().to_string())
print("non training rows dropped:", all_windows.loc[~keep, "dataset"].value_counts().to_dict() or 0)
print("training clips:", windows["clip_id"].nunique(), "sequence shape:", X.shape, "missing mask in npz:", "missing" in sequences.files)
print("lstm settings:", LSTM)

Outer folds. This table must match the `folds:` table printed by `scripts/04_cross_validate.py` row for row. If it differs, install the scikit-learn version printed by the local `.venv` (`pip install scikit-learn==<version>`), restart and rerun.

In [ ]:
def outer_splits(windows):
    splitter = StratifiedGroupKFold(n_splits=EVAL["outer_folds"], shuffle=True, random_state=SEED)
    return list(splitter.split(windows, windows["label"], windows["clip_id"]))


def fold_counts(fold, train, test):
    counts = {"fold": fold}
    for side, part in (("train", train), ("test", test)):
        clips = part["clip_id"].unique()
        counts[f"{side}_windows"] = len(part)
        counts[f"{side}_pos_windows"] = int(part["label"].sum())
        counts[f"{side}_clips"] = len(clips)
        counts[f"{side}_fall_clips"] = int(truth.loc[clips].sum())
    return counts


folds = outer_splits(windows)
counts = []
for fold, (train_index, test_index) in enumerate(folds, 1):
    train, test = windows.iloc[train_index], windows.iloc[test_index]
    if set(train["clip_id"]) & set(test["clip_id"]):
        raise AssertionError(f"fold {fold} has clips on both sides")
    counts.append(fold_counts(fold, train, test))
print(pd.DataFrame(counts).to_string(index=False))

In [ ]:
JOINT_SWAP = [1, 0, 3, 2, 5, 4, 7, 6, 9, 8, 11, 10]
FLIP_CHANNELS = np.array([2 * joint + axis for joint in JOINT_SWAP for axis in (0, 1)])


def model_input(X, missing):
    return np.concatenate([X, missing.astype(np.float32)], axis=-1)


def flip_horizontal(X, missing):
    flipped = X[..., FLIP_CHANNELS].copy()
    flipped[..., 0::2] *= -1
    return flipped, missing[..., FLIP_CHANNELS]


def augment(X, missing, y, rng):
    flipped, flipped_missing = flip_horizontal(X, missing)
    noise = rng.normal(0.0, LSTM["noise_std"], flipped.shape)
    noisy = np.where(flipped_missing, 0.0, flipped + noise).astype(np.float32)
    inputs = model_input(np.concatenate([X, noisy]), np.concatenate([missing, flipped_missing]))
    return inputs, np.concatenate([y, y])


def class_weight(y):
    counts = np.bincount(y, minlength=2)
    return {label: float(len(y) / (2 * count)) for label, count in enumerate(counts)}


def build_lstm(units, steps, channels):
    keras.utils.set_random_seed(SEED)
    model = keras.Sequential(
        [
            keras.Input((steps, channels)),
            keras.layers.LSTM(units),
            keras.layers.Dropout(LSTM["dropout"]),
            keras.layers.Dense(1, activation="sigmoid"),
        ]
    )
    model.compile(optimizer=keras.optimizers.Adam(LSTM["learning_rate"]), loss="binary_crossentropy")
    return model


def predict(model, inputs):
    return model.predict(inputs, verbose=0).ravel()


def recall_threshold(y_true, scores, target):
    positive = scores[y_true == 1]
    candidates = np.unique(positive)[::-1]
    recalls = np.array([(positive >= candidate).mean() for candidate in candidates])
    return float(candidates[np.argmax(recalls >= target)])


def ratio(numerator, denominator):
    return float(numerator / denominator) if denominator else float("nan")


def binary_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "recall": ratio(tp, tp + fn),
        "precision": ratio(tp, tp + fp),
        "specificity": ratio(tn, tn + fp),
        "f1": ratio(2 * tp, 2 * tp + fp + fn),
        "accuracy": ratio(tp + tn, tp + tn + fp + fn),
        "n_test": int(tp + tn + fp + fn),
        "n_pos": int(tp + fn),
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }


def clip_level(clip_ids, y_pred):
    fired = pd.Series(np.asarray(y_pred), index=np.asarray(clip_ids)).groupby(level=0).max()
    return truth.loc[fired.index].to_numpy(), fired.to_numpy()


def score_rows(test, predicted, base):
    clip_true, clip_pred = clip_level(test["clip_id"], predicted)
    return [
        {**base, "level": "window", **binary_metrics(test["label"].to_numpy(), predicted)},
        {**base, "level": "clip", **binary_metrics(clip_true, clip_pred)},
    ]


def validation_split(train_index):
    train = windows.iloc[train_index]
    splitter = StratifiedGroupKFold(n_splits=LSTM["val_splits"], shuffle=True, random_state=SEED)
    fit_local, val_local = next(splitter.split(train, train["label"], train["clip_id"]))
    fit_index, val_index = train_index[fit_local], train_index[val_local]
    if set(windows.loc[fit_index, "clip_id"]) & set(windows.loc[val_index, "clip_id"]):
        raise AssertionError("validation split shares clips with the fitting part")
    if y[val_index].sum() == 0:
        raise ValueError("validation split has no positive windows")
    return fit_index, val_index


def fit_lstm(units, fit_index, val_index):
    rng = np.random.default_rng(SEED)
    fit_inputs, fit_y = augment(X[fit_index], missing[fit_index], y[fit_index], rng)
    val_inputs, val_y = model_input(X[val_index], missing[val_index]), y[val_index]
    model = build_lstm(units, fit_inputs.shape[1], fit_inputs.shape[2])
    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=LSTM["patience"], restore_best_weights=True)
    model.fit(
        fit_inputs,
        fit_y,
        validation_data=(val_inputs, val_y),
        epochs=LSTM["epochs"],
        batch_size=LSTM["batch_size"],
        class_weight=class_weight(fit_y),
        callbacks=[stop],
        shuffle=True,
        verbose=0,
    )
    val_scores = predict(model, val_inputs)
    threshold = recall_threshold(val_y, val_scores, EVAL["recall_target"])
    val_f1 = binary_metrics(val_y, (val_scores >= threshold).astype(int))["f1"]
    return model, threshold, val_f1

Per fold: train one LSTM per units value on the augmented fitting part, pick the threshold on the clip grouped validation part, then score the test fold once. `default` uses `default_units`, `tuned` uses the units with the best validation F1 at its recall target threshold.

In [ ]:
rows = []
for fold, (train_index, test_index) in enumerate(folds, 1):
    fit_index, val_index = validation_split(train_index)
    trained = {units: fit_lstm(units, fit_index, val_index) for units in LSTM["units_grid"]}
    best_units = max(LSTM["units_grid"], key=lambda units: trained[units][2])
    test = windows.iloc[test_index]
    test_scores = {units: predict(trained[units][0], model_input(X[test_index], missing[test_index])) for units in {LSTM["default_units"], best_units}}
    for variant, units in (("default", LSTM["default_units"]), ("tuned", best_units)):
        threshold = trained[units][1]
        predicted = (test_scores[units] >= threshold).astype(int)
        base = {"detector": "lstm", "variant": variant, "fold": fold, "threshold": threshold, "best_params": str({"units": units})}
        rows += score_rows(test, predicted, base)
    validation_f1 = {units: round(trained[units][2], 3) for units in LSTM["units_grid"]}
    print(f"fold {fold}: fit {len(fit_index)} windows, validation {len(val_index)} windows, validation F1 {validation_f1}, tuned units {best_units}")

results = pd.DataFrame(rows)
results.insert(2, "level", results.pop("level"))
results.to_csv("lstm_cv_results.csv", index=False)
print("keep this file as results/tables/lstm_cv_results.csv, separate from cv_results.csv")
print(results.groupby(["level", "variant"])[["recall", "precision", "f1", "threshold"]].mean().round(3).to_string())
print("rows written:", len(results))

In [ ]:
from google.colab import files

files.download("lstm_cv_results.csv")